In [0]:
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings("ignore")

spark.sql("USE loan_risk")
print("Config loaded")

In [0]:
df_spark = spark.table("loan_risk.gold_loans_scored")
print(f" Rows   : {df_spark.count():,}")
print(f" Columns: {len(df_spark.columns)}")

In [0]:
print("Converting to Pandas...")
pdf = df_spark.toPandas()
print(f" Rows: {len(pdf):,}")

In [0]:
state_stats = (
    pdf.groupby("addr_state")
    .agg(
        loan_count       = ("default", "count"),
        default_rate     = ("default", "mean"),
        avg_int_rate     = ("int_rate", "mean"),
        avg_loan_amnt    = ("loan_amnt", "mean"),
        avg_default_prob = ("default_probability", "mean"),
    )
    .reset_index()
    .rename(columns={"addr_state": "state"})
    .sort_values("default_rate", ascending=False)
)

# Only states with 200+ loans to avoid noise
state_stats = state_stats[state_stats["loan_count"] >= 200]

print(f" States analyzed: {len(state_stats)}")
print("\nTop 5 Highest Default States:")
print(state_stats.head()[["state","loan_count","default_rate","avg_int_rate"]].to_string(index=False))

In [0]:
top15    = state_stats.head(15)
bottom15 = state_stats.tail(15).sort_values("default_rate")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Loan Default Rate by US State", fontsize=15, fontweight="bold")

# Highest default states
bars1 = axes[0].barh(
    top15["state"],
    top15["default_rate"] * 100,
    color="#d73027", edgecolor="white"
)
axes[0].set_title("Top 15 Highest Default States", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Default Rate (%)")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars1, top15["default_rate"]):
    axes[0].text(val * 100 + 0.1, bar.get_y() + bar.get_height()/2,
                 f"{val:.1%}", va="center", fontsize=9)

# Lowest default states
bars2 = axes[1].barh(
    bottom15["state"],
    bottom15["default_rate"] * 100,
    color="#1a9850", edgecolor="white"
)
axes[1].set_title("Top 15 Lowest Default States", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Default Rate (%)")
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars2, bottom15["default_rate"]):
    axes[1].text(val * 100 + 0.1, bar.get_y() + bar.get_height()/2,
                 f"{val:.1%}", va="center", fontsize=9)

plt.tight_layout()
plt.show()
print(" Chart 1 done")

In [0]:
grade_stats = (
    pdf.groupby("grade")
    .agg(
        loan_count       = ("default", "count"),
        default_rate     = ("default", "mean"),
        avg_int_rate     = ("int_rate", "mean"),
        avg_default_prob = ("default_probability", "mean"),
    )
    .reset_index()
    .sort_values("grade")
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Loan Default Risk by Grade", fontsize=15, fontweight="bold")

# Chart 1: Default rate by grade
colors = ["#1a9850","#91cf60","#d9ef8b","#fee08b","#fc8d59","#d73027","#a50026"]
bars = axes[0].bar(
    grade_stats["grade"],
    grade_stats["default_rate"] * 100,
    color=colors, edgecolor="white", width=0.5
)
axes[0].set_title("Default Rate by Grade", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Loan Grade (A=Best, G=Worst)")
axes[0].set_ylabel("Default Rate (%)")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, grade_stats["default_rate"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val:.1%}", ha="center", fontsize=10, fontweight="bold")

# Chart 2: Actual vs ML model prediction by grade
x = range(len(grade_stats))
width = 0.35
axes[1].bar([i - width/2 for i in x], grade_stats["default_rate"] * 100,
            width, label="Actual Default Rate", color="#d73027", alpha=0.8)
axes[1].bar([i + width/2 for i in x], grade_stats["avg_default_prob"] * 100,
            width, label="ML Model Prediction", color="#4393c3", alpha=0.8)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(grade_stats["grade"])
axes[1].set_xlabel("Loan Grade")
axes[1].set_ylabel("Default Rate / Probability (%)")
axes[1].set_title("Actual vs ML Prediction by Grade", fontsize=12, fontweight="bold")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.tight_layout()
plt.show()
print(" Chart 2 done")

In [0]:
purpose_stats = (
    pdf.groupby("purpose")
    .agg(
        loan_count       = ("default", "count"),
        default_rate     = ("default", "mean"),
        avg_loan_amnt    = ("loan_amnt", "mean"),
        avg_default_prob = ("default_probability", "mean"),
    )
    .reset_index()
    .sort_values("default_rate", ascending=False)
)

# Only purposes with 500+ loans
purpose_stats = purpose_stats[purpose_stats["loan_count"] >= 500]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Loan Default Risk by Purpose", fontsize=15, fontweight="bold")

# Chart 1: Default rate by purpose
avg_default = pdf["default"].mean()
colors_purpose = [
    "#d73027" if r > 0.25 else
    "#fc8d59" if r > 0.20 else
    "#1a9850"
    for r in purpose_stats["default_rate"]
]
bars = axes[0].barh(
    purpose_stats["purpose"],
    purpose_stats["default_rate"] * 100,
    color=colors_purpose, edgecolor="white", height=0.6
)
axes[0].axvline(avg_default * 100, color="black", linestyle="--",
                lw=1.5, label=f"Overall avg ({avg_default:.1%})")
axes[0].set_title("Default Rate by Purpose\n(Red >25%, Orange >20%, Green <20%)",
                  fontsize=11, fontweight="bold")
axes[0].set_xlabel("Default Rate (%)")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, purpose_stats["default_rate"]):
    axes[0].text(val * 100 + 0.2, bar.get_y() + bar.get_height()/2,
                 f"{val:.1%}", va="center", fontsize=9)
axes[0].legend()

# Chart 2: Bubble chart — default rate vs loan amount vs volume
scatter = axes[1].scatter(
    purpose_stats["avg_loan_amnt"],
    purpose_stats["default_rate"] * 100,
    s=purpose_stats["loan_count"] / 30,
    c=purpose_stats["default_rate"],
    cmap="RdYlGn_r",
    alpha=0.8,
    edgecolors="gray",
    linewidths=0.5
)
for _, row in purpose_stats.iterrows():
    axes[1].annotate(
        row["purpose"].replace("_", " ").title(),
        (row["avg_loan_amnt"], row["default_rate"] * 100),
        textcoords="offset points", xytext=(6, 4), fontsize=8
    )
plt.colorbar(scatter, ax=axes[1], label="Default Rate")
axes[1].set_xlabel("Average Loan Amount ($)")
axes[1].set_ylabel("Default Rate (%)")
axes[1].set_title("Default Rate vs Loan Amount\n(bubble size = loan volume)",
                  fontsize=11, fontweight="bold")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()
print(" Chart 3 done")

In [0]:
summary = spark.sql("""
    SELECT
        grade,
        COUNT(*)                                      AS total_loans,
        ROUND(AVG(default) * 100, 1)                  AS default_rate_pct,
        ROUND(AVG(int_rate), 1)                       AS avg_interest_rate,
        ROUND(AVG(loan_amnt), 0)                      AS avg_loan_amount,
        ROUND(AVG(default_probability) * 100, 1)      AS avg_model_score_pct,
        SUM(CASE WHEN risk_label = 'Very High' 
            THEN 1 ELSE 0 END)                        AS very_high_risk_count,
        SUM(CASE WHEN risk_label = 'High' 
            THEN 1 ELSE 0 END)                        AS high_risk_count
    FROM loan_risk.gold_loans_scored
    GROUP BY grade
    ORDER BY grade
""")

print("📋 Executive Summary by Grade:")
display(summary)

In [0]:
print("=" * 55)
print("📊 KEY FINDINGS — Loan Default Risk Analysis")
print("=" * 55)

# State findings
worst_state = state_stats.iloc[0]
best_state  = state_stats.iloc[-1]
print(f"\n📍 BY STATE:")
print(f"   Highest default : {worst_state['state']} ({worst_state['default_rate']:.1%})")
print(f"   Lowest default  : {best_state['state']} ({best_state['default_rate']:.1%})")
print(f"   Difference      : {worst_state['default_rate'] - best_state['default_rate']:.1%}")

# Grade findings
print(f"\n🏷️  BY GRADE:")
print(f"   Grade A default : 6.0%")
print(f"   Grade G default : 49.9%")
print(f"   Difference      : 8.3x higher for Grade G")

# Purpose findings
worst_purpose = purpose_stats.iloc[0]
best_purpose  = purpose_stats.iloc[-1]
print(f"\n🎯 BY PURPOSE:")
print(f"   Riskiest : {worst_purpose['purpose']} ({worst_purpose['default_rate']:.1%})")
print(f"   Safest   : {best_purpose['purpose']} ({best_purpose['default_rate']:.1%})")
print(f"   Difference: {worst_purpose['default_rate'] - best_purpose['default_rate']:.1%}")

print("\n" + "=" * 55)
print(" Insights notebook complete!")
print("=" * 55)